## **TEXT TO SQL LLM** (Qwen2.5-Coder-3B + LoRA on `gretelai/synthetic_text_to_sql`)

Pipeline: baseline execution-accuracy -> LoRA (completion-only loss) -> same eval after -> save adapter to Drive.

In [1]:
!pip install -q -U transformers trl peft accelerate bitsandbytes datasets "torchao>=0.16.0"
# After this cell finishes: Runtime > Restart session, then run from the next cell (do NOT re-run this one).

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 140.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 43.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
import transformers, trl, peft, torch
print(transformers.__version__, trl.__version__, peft.__version__, torch.__version__)

5.19.0 1.15.0 0.21.2 2.11.0+cu130


In [3]:
import torch
assert torch.cuda.is_available(), "Enable a GPU: Runtime > Change runtime type"

# L4 / A100 support bf16 (more stable). T4 does not -> fall back to fp16.
USE_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(torch.cuda.get_device_name(0), "| bf16:", USE_BF16)

NVIDIA L4 | bf16: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')          # files live under /content/drive/MyDrive/...

Mounted at /content/drive


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype=DTYPE)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"       # training side; generate_batch() switches to "left" temporarily

model.config.use_cache = False

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [6]:
messages = [{"role": "user", "content": "Who are you?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

I am Qwen, an AI model created by Alibaba Cloud. I am designed to assist with a wide range of tasks and provide information on various topics. How can I help you today?


In [7]:
import re

SYSTEM = ("You are a SQL assistant. Given a table schema and a question, "
          "reply with ONLY the SQL query, nothing else.")

# Keep only CREATE statements for the prompt (drop INSERT sample rows) -> shorter prompts,
# and it matches real usage where you only give the model the schema.
def schema_only(context: str) -> str:
    stmts = [s.strip() for s in context.split(";") if s.strip()]
    creates = [s + ";" for s in stmts if s.upper().startswith("CREATE")]
    return "\n".join(creates) if creates else context

def build_messages(schema: str, question: str):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Schema:\n{schema}\n\nQuestion: {question}"},
    ]

def build_prompt(schema: str, question: str) -> str:
    return tokenizer.apply_chat_template(
        build_messages(schema, question), tokenize=False, add_generation_prompt=True
    )

def clean_sql(text: str) -> str:
    text = re.sub(r"^```(?:sql)?\s*|\s*```$", "", text.strip(), flags=re.I)
    return text.strip()

In [8]:
@torch.no_grad()
def generate_batch(prompts, max_new_tokens=200, batch_size=16):
    outs = []
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"            # required for batched generation
    model.eval()
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True).to(model.device)
        ids = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            use_cache=True, pad_token_id=tokenizer.pad_token_id,
        )
        new = ids[:, enc["input_ids"].shape[1]:]
        outs += [t.strip() for t in tokenizer.batch_decode(new, skip_special_tokens=True)]
    tokenizer.padding_side = old_side
    return outs

def generate(prompt: str, max_new_tokens: int = 200) -> str:
    return generate_batch([prompt], max_new_tokens, batch_size=1)[0]

# quick test
print(generate(build_prompt(
    "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    "List the names of employees in the Engineering department earning more than 100000.",
)))

SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000;


In [9]:
from datasets import load_dataset

raw = load_dataset("gretelai/synthetic_text_to_sql")
print(raw)
row = raw["train"][0]
print({k: row[k] for k in ["sql_prompt", "sql_context", "sql"]})   # check: does sql_context contain INSERTs?

README.md:   0%|          | 0.00/8.19k [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…): reconstructing file:   0%|          |  0.00B / 32.4MB            

synthetic_text_to_sql_train.snappy.parqu(…): downloading bytes:           |  0.00B            

synthetic_text_to_sql_test.snappy.parque(…): reconstructing file:   0%|          |  0.00B / 1.90MB            

synthetic_text_to_sql_test.snappy.parque(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 100000
    })
    test: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 5851
    })
})
{'sql_prompt': 'What is the total volume of timber sold by each salesperson, sorted by salesperson?', 'sql_context': "CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date)

In [10]:
import sqlite3
from collections import Counter

# ---- Execution helpers (run SQL on an in-memory SQLite built from sql_context) ----
def run_sql(context: str, query: str, max_rows: int = 1000):
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(context)
        return con.execute(query).fetchmany(max_rows)
    except Exception:
        return None
    finally:
        con.close()

def _norm_rows(rows):
    return [tuple(round(x, 4) if isinstance(x, float) else x for x in r) for r in rows]

def same_result(gold, pred, ordered: bool) -> bool:
    if gold is None or pred is None:
        return False
    g, p = _norm_rows(gold), _norm_rows(pred)
    return g == p if ordered else Counter(g) == Counter(p)

# Keep only examples whose gold SQL actually runs on SQLite and returns rows
# (drops dialect-specific / broken labels -> cleaner training + a fair eval).
def executable_gold(r) -> bool:
    out = run_sql(r["sql_context"], r["sql"])
    return out is not None and len(out) > 0

SEED, N_TRAIN, N_VAL, N_TEST = 42, 8000, 200, 200

pool = raw["train"].shuffle(seed=SEED).select(range(N_TRAIN + N_VAL + 3000))
pool = pool.filter(executable_gold)
val_raw   = pool.select(range(N_VAL))
train_raw = pool.select(range(N_VAL, min(len(pool), N_VAL + N_TRAIN)))

test_pool = raw["test"].shuffle(seed=SEED).select(range(N_TEST * 3)).filter(executable_gold)
test_raw  = test_pool.select(range(min(N_TEST, len(test_pool))))

print("Train:", len(train_raw), "| Val (loss):", len(val_raw), "| Test (exec acc):", len(test_raw))

Filter:   0%|          | 0/11200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/600 [00:00<?, ? examples/s]

Train: 6330 | Val (loss): 200 | Test (exec acc): 200


In [11]:
def execution_accuracy(ds, max_new_tokens=200, show_fails=3):
    prompts = [build_prompt(schema_only(r["sql_context"]), r["sql_prompt"]) for r in ds]
    preds = [clean_sql(p) for p in generate_batch(prompts, max_new_tokens)]

    correct = valid = 0
    fails = []
    for r, pred in zip(ds, preds):
        gold = run_sql(r["sql_context"], r["sql"])
        out  = run_sql(r["sql_context"], pred)
        valid += out is not None
        ok = same_result(gold, out, ordered="order by" in r["sql"].lower())
        correct += ok
        if not ok and len(fails) < show_fails:
            fails.append((r["sql_prompt"], r["sql"], pred))

    n = len(ds)
    print(f"Execution accuracy: {correct/n:.1%} ({correct}/{n}) | valid SQL: {valid/n:.1%}")
    for q, g, p in fails:
        print("\n[FAIL] Q   :", q, "\n       GOLD:", g, "\n       PRED:", p)
    return {"exec_acc": correct / n, "valid": valid / n, "preds": preds}

print("=" * 70, "\nBASELINE (before fine-tuning)\n" + "=" * 70)
base_res = execution_accuracy(test_raw)

BASELINE (before fine-tuning)
Execution accuracy: 57.0% (114/200) | valid SQL: 89.5%

[FAIL] Q   : List the number of cultural heritage sites in Tokyo and Seoul. 
       GOLD: SELECT city, COUNT(*) FROM asian_sites WHERE city IN ('Tokyo', 'Seoul') AND type = 'historical' GROUP BY city; 
       PRED: SELECT COUNT(*) AS heritage_count, city FROM asian_sites WHERE type = 'Cultural Heritage' AND city IN ('Tokyo', 'Seoul') GROUP BY city

[FAIL] Q   : Calculate the percentage of mobile and broadband subscribers in each region. 
       GOLD: SELECT r.region_name, (COUNT(m.subscriber_id) * 100.0 / (COUNT(m.subscriber_id) + COUNT(b.subscriber_id))) AS mobile_percentage, (COUNT(b.subscriber_id) * 100.0 / (COUNT(m.subscriber_id) + COUNT(b.subscriber_id))) AS broadband_percentage FROM regions r LEFT JOIN mobile_subscribers m ON r.region_id = m.region_id LEFT JOIN broadband_subscribers b ON r.region_id = b.region_id GROUP BY r.region_id; 
       PRED: SELECT 
    r.region_name,
    ROUND((SUM(CASE 

In [12]:
# Qualitative probes (same ones as before, reused after training)
PROBES = [
    {"schema": "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
     "question": "List the names of employees in the Engineering department earning more than 100000."},
    {"schema": "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
     "question": "What is the total order amount per customer in 2024?"},
    {"schema": "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
     "question": "Show the top 5 highest rated horror movies released after 2015."},
]
base_outputs = generate_batch([build_prompt(p["schema"], p["question"]) for p in PROBES])
for i, (p, a) in enumerate(zip(PROBES, base_outputs), 1):
    print(f"--- Probe {i} ---\nQ: {p['question']}\nA: {a}\n")

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000;

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: SELECT customer_id, SUM(amount) AS total_amount
FROM orders
WHERE YEAR(order_date) = 2024
GROUP BY customer_id;

--- Probe 3 ---
Q: Show the top 5 highest rated horror movies released after 2015.
A: SELECT title, rating FROM movies WHERE genre = 'Horror' AND year > 2015 ORDER BY rating DESC LIMIT 5



In [13]:
# prompt/completion format -> TRL masks the prompt and trains the loss on the SQL answer only
def to_prompt_completion(row):
    return {
        "prompt":     build_messages(schema_only(row["sql_context"]), row["sql_prompt"]),
        "completion": [{"role": "assistant", "content": row["sql"].strip()}],
    }

train_ds = train_raw.map(to_prompt_completion, remove_columns=train_raw.column_names)
val_ds   = val_raw.map  (to_prompt_completion, remove_columns=val_raw.column_names)
print(train_ds[0])

Map:   0%|          | 0/6330 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

{'prompt': [{'role': 'system', 'content': 'You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.'}, {'role': 'user', 'content': 'Schema:\nCREATE TABLE marine_life (life_id INT, life_name VARCHAR(50), region VARCHAR(50));\n\nQuestion: What is the total number of marine species in the Arctic region?'}], 'completion': [{'role': 'assistant', 'content': "SELECT COUNT(*) FROM marine_life WHERE region = 'Arctic';"}]}


In [14]:
from peft import LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # needed because base params are frozen

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,                     # alpha / r = 2.0
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",       # attention + MLP (q,k,v,o,gate,up,down); excludes lm_head
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [15]:
from trl import SFTTrainer, SFTConfig

# Checkpoints go to Drive so a Colab disconnect doesn't lose the run
OUTPUT_DIR = "/content/drive/MyDrive/text2sql-qwen-coder-lora/checkpoints"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=1e-4,                # lower than 2e-4: Coder model is already strong at SQL
    lr_scheduler_type="cosine",
    warmup_steps=20,
    optim="adamw_torch",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    max_length=768,
    completion_only_loss=True,         # loss on the SQL answer only
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

# Sanity check: the text the loss is computed on should be ONLY the SQL answer
batch = next(iter(trainer.get_train_dataloader()))
ids, lab = batch["input_ids"][0], batch["labels"][0]
print("Trained-on text:", repr(tokenizer.decode(ids[lab != -100])))

Tokenizing train dataset:   0%|          | 0/6330 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6330 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6330 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/6330 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Trained-on text: 'SELECT AVG(quantity) FROM Dishes WHERE has_meat = true;<|im_end|>\n'


In [16]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,0.196652,0.195789,0.179688,0.940020,181756.000000
200,0.197384,0.191445,0.178677,0.943013,365503.000000
300,0.175572,0.185868,0.182912,0.941835,550633.000000
396,0.174760,0.185189,0.178764,0.942803,724105.000000


TrainOutput(global_step=396, training_loss=0.1904265730067937, metrics={'train_runtime': 2040.3908, 'train_samples_per_second': 3.102, 'train_steps_per_second': 0.194, 'total_flos': 1.3543135150030848e+16, 'train_loss': 0.1904265730067937, 'epoch': 1.0})

In [17]:
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 6.45 GB


In [18]:
model.config.use_cache = True
model.eval()

print("=" * 70, "\nFINE-TUNED (after LoRA)\n" + "=" * 70)
ft_res = execution_accuracy(test_raw)

print("\n" + "=" * 70)
print(f"{'':12}{'Exec acc':>10}{'Valid SQL':>11}")
print(f"{'Base':12}{base_res['exec_acc']:>10.1%}{base_res['valid']:>11.1%}")
print(f"{'Fine-tuned':12}{ft_res['exec_acc']:>10.1%}{ft_res['valid']:>11.1%}")

FINE-TUNED (after LoRA)
Execution accuracy: 69.5% (139/200) | valid SQL: 97.5%

[FAIL] Q   : List the number of cultural heritage sites in Tokyo and Seoul. 
       GOLD: SELECT city, COUNT(*) FROM asian_sites WHERE city IN ('Tokyo', 'Seoul') AND type = 'historical' GROUP BY city; 
       PRED: SELECT city, COUNT(*) FROM asian_sites WHERE type = 'Cultural Heritage' GROUP BY city;

[FAIL] Q   : Calculate the percentage of mobile and broadband subscribers in each region. 
       GOLD: SELECT r.region_name, (COUNT(m.subscriber_id) * 100.0 / (COUNT(m.subscriber_id) + COUNT(b.subscriber_id))) AS mobile_percentage, (COUNT(b.subscriber_id) * 100.0 / (COUNT(m.subscriber_id) + COUNT(b.subscriber_id))) AS broadband_percentage FROM regions r LEFT JOIN mobile_subscribers m ON r.region_id = m.region_id LEFT JOIN broadband_subscribers b ON r.region_id = b.region_id GROUP BY r.region_id; 
       PRED: SELECT r.region_name, (COUNT(CASE WHEN ms.subscriber_id IS NOT NULL THEN 1 END) * 100.0 / COUNT(r.reg

In [19]:
ft_outputs = generate_batch([build_prompt(p["schema"], p["question"]) for p in PROBES])
for i, p in enumerate(PROBES, 1):
    print(f"--- Probe {i} ---\nQ:      {p['question']}")
    print(f"BEFORE: {base_outputs[i-1]}\n{'-'*70}\nAFTER : {ft_outputs[i-1]}\n{'='*70}")

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000;
----------------------------------------------------------------------
AFTER : SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000;
--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: SELECT customer_id, SUM(amount) AS total_amount
FROM orders
WHERE YEAR(order_date) = 2024
GROUP BY customer_id;
----------------------------------------------------------------------
AFTER : SELECT customer_id, SUM(amount) as total_order_amount FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY customer_id;
--- Probe 3 ---
Q:      Show the top 5 highest rated horror movies released after 2015.
BEFORE: SELECT title, rating FROM movies WHERE genre = 'Horror' AND year > 2015 ORDER BY rating DESC LIMIT 5
---------------------------

In [ ]:
import os

ADAPTER_DIR = "/content/drive/MyDrive/text-to-sql-lora-adapter"  
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

total = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
            for f in os.listdir(ADAPTER_DIR) if os.path.isfile(os.path.join(ADAPTER_DIR, f)))
print(f"Saved to {ADAPTER_DIR} | size on disk: {total/1024**2:.2f} MB")

Saved to /content/drive/MyDrive/text-to-sql-lora-adapter | size on disk: 125.15 MB
